In [12]:
import pandas as pd 

In [13]:
df = pd.read_json("r_SuicideWatch_posts_large.jsonl" ,encoding="utf-8",lines =True)

In [14]:
df.head(15)

,_meta,all_awardings,allow_live_comments,approved_at_utc,approved_by,archived,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,...,preview,author_cakeday,post_hint,link_flair_template_id,location_lat,location_long,location_name,websocket_url,num_duplicates,profile_img
0,{'retrieved_2nd_on': 1767335750},[],False,NaN,NaN,False,catcatchesmouse,NaN,modmsg,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,{'retrieved_2nd_on': 1767336008},[],False,NaN,NaN,False,KateLoop43,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,{'retrieved_2nd_on': 1767336062},[],False,NaN,NaN,False,[deleted],,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,{'retrieved_2nd_on': 1767336070},[],False,NaN,NaN,False,PublicAd6378,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,{'retrieved_2nd_on': 1767336128},[],False,NaN,NaN,False,Left_Grapefruit5205,NaN,modmsg,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,{'retrieved_2nd_on': 1767336235},[],False,NaN,NaN,False,King_Wolf2099,NaN,modmsg,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"{'removal_type': 'deleted', 'retrieved_2nd_on'...",[],False,NaN,NaN,False,Onlylonely5,NaN,modmsg,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,"{'is_edited': True, 'removal_type': 'reddit', ...",[],False,NaN,NaN,False,Fearless_Group1546,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,"{'removal_type': 'deleted', 'retrieved_2nd_on'...",[],False,NaN,NaN,False,AbsoluteBingo,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,{'retrieved_2nd_on': 1767337307},[],False,NaN,NaN,False,Solid-Cantaloupe-635,NaN,modmsg,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
counts = df.groupby('author').size()
counts[counts>2]
result = df[df['author'].isin(counts[counts > 2].index)]

In [16]:
result[['title','selftext']]

,title,selftext
2,HAPPY JEW YEAR GUYS,[removed]
4,Happy new year 🎊 to all,[removed]
5,Hopefully next year will be my last.,"After being alive for 23 years, i find life to..."
6,I’m suicidal,I self harm almost everyday. I have been abuse...
9,Sick,Running an extremely high fever and I feel lik...
...,...,...
125816,Being a trans male is the most cucked shit ever,"I will never have a penis, i have been forcibl..."
125819,I'm so tired,I haven't gone to school for more than a month...
125820,What should I do before I go?,I don’t really have anyone in my life. Not rea...
125827,Stuck,It's super fun not knowing what is real or a s...


In [17]:
result.query("selftext != '[removed]'").index

Index([     5,      6,      9,     13,     14,     15,     24,     27,     28,
           29,
       ...
       125797, 125803, 125804, 125808, 125809, 125811, 125816, 125819, 125820,
       125827],
      dtype='int64', length=43958)

In [18]:
intermediary_df = result.query("selftext != '[removed]' and selftext.str.len() > 2")[['author','title','selftext']]

In [19]:
intermediary_df

,author,title,selftext
5,King_Wolf2099,Hopefully next year will be my last.,"After being alive for 23 years, i find life to..."
6,Onlylonely5,I’m suicidal,I self harm almost everyday. I have been abuse...
9,Solid-Cantaloupe-635,Sick,Running an extremely high fever and I feel lik...
13,StarDustJinx,i don't want to or plan to enter 2026. i'm jus...,everything in my life is making my episodes an...
14,quirktronic,"Best Friend hates me, Girlfriend hates me, jus...",What the title says. I currently feel no effec...
...,...,...,...
125811,BumBillBee,How the hell do people go on.,"Brief back story, grew up in material comfort ..."
125816,No_Ticket5416,Being a trans male is the most cucked shit ever,"I will never have a penis, i have been forcibl..."
125819,Personal-Flower-8287,I'm so tired,I haven't gone to school for more than a month...
125820,Particular_Steak_485,What should I do before I go?,I don’t really have anyone in my life. Not rea...


In [20]:
import re
from tqdm import tqdm
tqdm.pandas()
def clean_and_spellcheck(text):
    if not isinstance(text, str):
        return []
    
    # Separate glued letters and numbers (e.g., '2929\n\n\nThat's' or 'mental14')
    text = re.sub(r'([a-zA-Z]+)(\d+)', r'\1 \2', text)
    
    # Normalize curly apostrophes to straight apostrophes
    text = text.replace("’", "'")
    
    # Extract clean alphabetic tokens
    clean_tokens = re.findall(r"[a-zA-Z]+(?:'[a-zA-Z]+)?", text.lower())
    
    return " ".join(clean_tokens)
intermediary_df['cleaned_selftext'] = intermediary_df['selftext'].progress_apply(clean_and_spellcheck)

100%|██████████| 43519/43519 [00:15<00:00, 2752.34it/s] 


In [21]:
vectorizer = joblib.load("tfidf_vectorizer.pkl")

X_new = vectorizer.transform(["Your momma is so fat, when she fell I didn't laugh, but the sidewalk cracked up."])

X_new_tensor = torch.tensor(X_new.toarray(), dtype=torch.float32)

with torch.no_grad():
    logits = model(X_new_tensor)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).float()

print("Probability:", probabilities.item())
print("Prediction:", predictions.item())


NameError: name 'joblib' is not defined

In [22]:
import joblib
import torch
import torch.nn as nn
from tqdm import tqdm

# Load vectorizer
vectorizer = joblib.load("tfidf_vectorizer.pkl")

input_size = len(vectorizer.get_feature_names_out())

tqdm.pandas()

model = nn.Sequential(
    nn.Linear(input_size, 16),
    nn.ReLU(),
    nn.Linear(16, 10),
    nn.ReLU(),
    nn.Linear(10, 1)
)

# Load trained weights
model.load_state_dict(
    torch.load(
        "suicide_classifier.pth",
        map_location="cpu",
        weights_only=True
    )
)

# Load model
model.load_state_dict(torch.load("suicide_classifier.pth", weights_only=True))
model.eval()

# new_text = ["""
# i just wanna give up. I'm tired.

# i don't have money but I guess I could find a way to get a bottle of silver cleaner. i just want everything to end.
# """]

def pred(new_text:str):
    X_new = vectorizer.transform([new_text])

    X_new_tensor = torch.tensor(X_new.toarray(), dtype=torch.float32)

    with torch.no_grad():
        logits = model(X_new_tensor)
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

    # print("Probability:", probabilities.item())
    
    return(probabilities.item())


intermediary_df['suicidal_text'] = intermediary_df['cleaned_selftext'].progress_apply(pred)

100%|██████████| 43519/43519 [00:36<00:00, 1185.66it/s]


In [23]:
intermediary_df.groupby('author').size()

author
--Tyler_durden-      2
--jyushimatsudesu    2
--piplup--           2
--turbulence--       3
-2wG                 6
                    ..
zombiemaxxer         4
zombit3z             4
zumodecarne          2
zy-zee               2
zzwolfy              4
Length: 8577, dtype: int64

In [24]:
intermediary_df.query("suicidal_text < 0.5")


,author,title,selftext,cleaned_selftext,suicidal_text
40,042004grr,Enjoying myself right now,At the club. God i love the club (but its just...,at the club god i love the club but its just l...,5.259817e-13
86,Warm_Front259,My family lack tact when talking about my suic...,My family keeps making me feel guilty about my...,my family keeps making me feel guilty about my...,4.696803e-01
104,Shin_Ollie,End Times,I'll he dead in January. All my freinds were f...,i'll he dead in january all my freinds were fa...,1.554600e-24
121,042004grr,Left the club,Damn i kinda went crazy crazy.. now im just de...,damn i kinda went crazy crazy now im just deal...,1.181575e-35
133,hateoryz,cutting through my body in familiar ways,i hate it,i hate it,1.893347e-02
...,...,...,...,...,...
125787,Olivetrees3228,Ny gf broke up wanna kill myself,I appreciate if anyone can text to me. I’m so ...,i appreciate if anyone can text to me i'm so a...,3.572333e-10
125803,Fine-Cut-9234,Hello,Cineva din Romania?,cineva din romania,2.408264e-01
125808,Aromatic-Abroad-2650,Please let me breath PLEASE LET ME BREATH PLEA...,I can't I seriously can't am weak I fucking ha...,i can't i seriously can't am weak i fucking ha...,5.558727e-06
125819,Personal-Flower-8287,I'm so tired,I haven't gone to school for more than a month...,i haven't gone to school for more than a month...,3.180527e-08


In [25]:
suicide_watch_large_df = pd.read_json("r_SuicideWatch_posts_large.jsonl", lines=True)

In [26]:
suicide_watch_large_df.count()

_meta                  121599
all_awardings          125830
allow_live_comments    125830
approved_at_utc             0
approved_by                 0
                        ...  
location_long               0
location_name               0
websocket_url           79343
num_duplicates            228
profile_img              5870
Length: 116, dtype: int64

In [27]:
intermediate_positive_cohort = suicide_watch_large_df.groupby("author").size()
result = suicide_watch_large_df[suicide_watch_large_df['author'].isin(intermediate_positive_cohort[intermediate_positive_cohort > 5].index)]

In [28]:
positive_cohort = result


In [29]:
positive_cohort[['author','selftext']]
positive_cohort["clean_selftext"] = positive_cohort['selftext'].progress_apply(clean_and_spellcheck)

100%|██████████| 31404/31404 [00:01<00:00, 18490.85it/s]


In [30]:
positive_cohort['initial_pred'] = positive_cohort['clean_selftext'].progress_apply(pred)

100%|██████████| 31404/31404 [00:22<00:00, 1389.02it/s]


In [31]:
(positive_cohort.sort_values('author'))[['author','clean_selftext','initial_pred','created_utc']]

,author,clean_selftext,initial_pred,created_utc
51212,-2wG,governments calling us all terrorists banning ...,1.000000,1778657821
96828,-2wG,who else is doing this another weekend flies b...,0.000934,1788850399
8249,-2wG,even my own spouse any time im suffering i can...,1.000000,1768961174
52227,-2wG,removed,0.992949,1778886479
90574,-2wG,f just dumb and useless barely holding it toge...,0.995810,1787538774
...,...,...,...,...
23132,zombie-b0yy,well i already attempted once but god im tired...,0.995097,1772313817
66298,zombie-b0yy,i already attempted twice but im just so tired...,1.000000,1782137925
32847,zombie-b0yy,i think i have adhd but i cant get diagnosed b...,0.077388,1774491784
30194,zombie-b0yy,i got an exam coming up nobody wants to talk t...,0.998521,1773882718


In [32]:
(positive_cohort.sort_values('author'))[['author','clean_selftext','initial_pred','created_utc']].query('initial_pred < 0.4')

,author,clean_selftext,initial_pred,created_utc
96828,-2wG,who else is doing this another weekend flies b...,9.335682e-04,1788850399
21007,-2wG,i should die im forced to work for them i shou...,2.280740e-01,1771817025
121518,-Dthrowaway-,i need to be chickened to stop this intrusive ...,8.365551e-03,1794526528
89352,-Dthrowaway-,i'm chronically ill i don't have much left in ...,5.581307e-12,1787270819
114949,-Dthrowaway-,i can't eat delicious things i'm in pain atlea...,5.927206e-14,1792890834
...,...,...,...,...
24238,zippobunny,i don't even know what to class this as i pick...,5.513472e-09,1772556874
70306,zippobunny,i wish someone would punch my stomach so many ...,7.821037e-13,1783047547
50914,zippobunny,i don't even really know what to say about thi...,1.865962e-01,1778596972
31145,zombie-b0yy,i just relapsed and im planning on just doing ...,2.314952e-05,1774114446


In [33]:
(positive_cohort.sort_values('author'))[['author','clean_selftext','initial_pred','created_utc']].query('initial_pred > 0.5')

,author,clean_selftext,initial_pred,created_utc
51212,-2wG,governments calling us all terrorists banning ...,1.000000,1778657821
8249,-2wG,even my own spouse any time im suffering i can...,1.000000,1768961174
52227,-2wG,removed,0.992949,1778886479
90574,-2wG,f just dumb and useless barely holding it toge...,0.995810,1787538774
33956,-2wG,ive got chronic fatigue no one knows why i'm a...,1.000000,1774762585
...,...,...,...,...
111679,zodiackkr19,i need to end it it feels really bad inside i ...,1.000000,1792030031
23132,zombie-b0yy,well i already attempted once but god im tired...,0.995097,1772313817
66298,zombie-b0yy,i already attempted twice but im just so tired...,1.000000,1782137925
30194,zombie-b0yy,i got an exam coming up nobody wants to talk t...,0.998521,1773882718


In [34]:
positive_cohort_filtered = positive_cohort[['subreddit', 'title', 'selftext', 'created_utc']].copy()
positive_cohort_filtered['created_utc'] = (
    pd.to_datetime(positive_cohort_filtered['created_utc'], unit='s', utc=True)
    .dt.tz_convert('Asia/Kolkata')
)

In [35]:
positive_cohort_filtered.query('title != "removed"')



,subreddit,title,selftext,created_utc
2,SuicideWatch,HAPPY JEW YEAR GUYS,[removed],2026-01-01 00:10:52+05:30
4,SuicideWatch,Happy new year 🎊 to all,[removed],2026-01-01 00:11:56+05:30
6,SuicideWatch,I’m suicidal,I self harm almost everyday. I have been abuse...,2026-01-01 00:19:52+05:30
13,SuicideWatch,i don't want to or plan to enter 2026. i'm jus...,everything in my life is making my episodes an...,2026-01-01 00:44:19+05:30
15,SuicideWatch,Before the clock strikes 12,"I have a everything i need to do it, just goin...",2026-01-01 00:50:07+05:30
...,...,...,...,...
125810,SuicideWatch,I want to be done.,[removed],2026-11-25 22:25:12+05:30
125811,SuicideWatch,How the hell do people go on.,"Brief back story, grew up in material comfort ...",2026-11-25 22:25:19+05:30
125816,SuicideWatch,Being a trans male is the most cucked shit ever,"I will never have a penis, i have been forcibl...",2026-11-25 22:50:56+05:30
125819,SuicideWatch,I'm so tired,I haven't gone to school for more than a month...,2026-11-25 22:58:15+05:30


In [36]:
positive_cohort_filtered['month'] = positive_cohort_filtered['created_utc'].dt.strftime('%Y-%m')

# Count posts per month
monthly_post_counts = positive_cohort_filtered.groupby('month').size()
print(monthly_post_counts)

month
2026-01    2822
2026-02    2677
2026-03    2974
2026-04    2848
2026-05    3154
2026-06    2838
2026-07    3148
2026-08    3281
2026-09    3156
2026-10    2596
2026-11    1910
dtype: int64


In [38]:
positive_cohort_filtered = positive_cohort[['author', 'subreddit', 'title', 'selftext', 'created_utc']].copy()
positive_cohort_filtered['created_utc'] = (
    pd.to_datetime(positive_cohort_filtered['created_utc'], unit='s', utc=True)
    .dt.tz_convert('Asia/Kolkata')
)

# 1. Get each user's first post timestamp (t0)
user_anchors = (
    positive_cohort_filtered.groupby('author', as_index=False)['created_utc']
    .min()
    .rename(columns={'created_utc': 't0'})
)

# 2. Extract month and count unique users per month
user_anchors['month'] = user_anchors['t0'].dt.strftime('%Y-%m')
monthly_user_quotas = user_anchors.groupby('month')['author'].count()

print(monthly_user_quotas)

month
2026-01    689
2026-02    336
2026-03    257
2026-04    234
2026-05    219
2026-06    179
2026-07    190
2026-08    167
2026-09    147
2026-10     79
2026-11     29
Name: author, dtype: int64


In [ ]:
#userId,firstdate(t0)

['_meta',
 'all_awardings',
 'allow_live_comments',
 'approved_at_utc',
 'approved_by',
 'archived',
 'author',
 'author_flair_background_color',
 'author_flair_css_class',
 'author_flair_richtext',
 'author_flair_template_id',
 'author_flair_text',
 'author_flair_text_color',
 'author_flair_type',
 'author_fullname',
 'author_is_blocked',
 'author_patreon_flair',
 'author_premium',
 'awarders',
 'banned_at_utc',
 'banned_by',
 'can_gild',
 'can_mod_post',
 'category',
 'clicked',
 'content_categories',
 'contest_mode',
 'created',
 'created_utc',
 'discussion_type',
 'distinguished',
 'domain',
 'downs',
 'edited',
 'gilded',
 'gildings',
 'hidden',
 'hide_score',
 'id',
 'is_created_from_ads_ui',
 'is_crosspostable',
 'is_meta',
 'is_original_content',
 'is_reddit_media_domain',
 'is_robot_indexable',
 'is_self',
 'is_video',
 'likes',
 'link_flair_background_color',
 'link_flair_css_class',
 'link_flair_richtext',
 'link_flair_text',
 'link_flair_text_color',
 'link_flair_type',
 'l

In [ ]:
user_df = pd.read_json('u_yaoimalover101_posts.jsonl',lines=True)

In [ ]:
user_df[['subreddit','title','selftext','created_utc']]

,subreddit,title,selftext,created_utc
0,Fitness,f15 new! needs advice,[removed],1684678275
1,introvert,school mean girls,so today in school while i was walking in the ...,1687362229
2,CharacterAI,ummmm,,1691453680
3,ugly,tired,[removed],1709154660
4,ugly,shhdhdj,[removed],1709155302
...,...,...,...,...
153,zepetoavatars,Since I am the only one here,,1792168268
154,zepetoavatars,Helloooo?,,1792168337
155,zepetoavatars,Hiii,Please,1792168381
156,zepetoavatars,Cute,,1792652873


In [ ]:
user_df[['subreddit','title','selftext','created_utc']].query("subreddit == 'SuicideWatch'")

,subreddit,title,selftext,created_utc
89,SuicideWatch,Idk,Where else or who can I talk to about feeling ...,1745796791
90,SuicideWatch,Idk,Does anyone else ever feel stuck in their body...,1747881615
96,SuicideWatch,This life sucks dealing with verbal abuse and ...,Just earlier my step dad charged at me full fo...,1753428925
101,SuicideWatch,Idk lol,"Lately, it’s been getting really bad again.\nT...",1760451580
110,SuicideWatch,Tired,"I shouldn’t be sad.\nI have food, water, and c...",1762369147
113,SuicideWatch,Lonely defeated,"Now my home situation is terrible, and there a...",1762741605
114,SuicideWatch,Sucks,Another day of interacting with these horrible...,1763222660
117,SuicideWatch,So tired,I'm tired kf this shit\nAnother garbage worthl...,1763558361
121,SuicideWatch,Life sucks as a unattractive woman and if you ...,Do guys only love pretty girls?\nDo they only ...,1764004046
123,SuicideWatch,Lonely and ( wanting to disappear),[removed],1764076592


In [ ]:
filtered_df = user_df[['subreddit', 'title', 'selftext', 'created_utc']].copy()
filtered_df['created_ist'] = (
    pd.to_datetime(filtered_df['created_utc'], unit='s', utc=True)
    .dt.tz_convert('Asia/Kolkata')
)

In [ ]:
filtered_df

,subreddit,title,selftext,created_utc,created_ist
0,Fitness,f15 new! needs advice,[removed],1684678275,2023-05-21 19:41:15+05:30
1,introvert,school mean girls,so today in school while i was walking in the ...,1687362229,2023-06-21 21:13:49+05:30
2,CharacterAI,ummmm,,1691453680,2023-08-08 05:44:40+05:30
3,ugly,tired,[removed],1709154660,2024-02-29 02:41:00+05:30
4,ugly,shhdhdj,[removed],1709155302,2024-02-29 02:51:42+05:30
...,...,...,...,...,...
153,zepetoavatars,Since I am the only one here,,1792168268,2026-10-16 22:01:08+05:30
154,zepetoavatars,Helloooo?,,1792168337,2026-10-16 22:02:17+05:30
155,zepetoavatars,Hiii,Please,1792168381,2026-10-16 22:03:01+05:30
156,zepetoavatars,Cute,,1792652873,2026-10-22 12:37:53+05:30


In [ ]:
steam_deck_df = pd.read_json('r_Steamdeck_posts.jsonl',lines=True)

In [ ]:
steam_deck_df[['subreddit','title','selftext','created_utc']]

,subreddit,title,selftext,created_utc
0,SteamDeck,r/SteamDeck Lounge,A place for members of r/SteamDeck to chat wit...,1626368162
1,SteamDeck,Valve Announces: Steam Deck,"Valve Announces the Steam Deck, a $400 Handhel...",1626368375
2,SteamDeck,Will the NVMe drive be user-replacable?,I know a lot of these handheld PCs have had us...,1626368820
3,SteamDeck,Steam Deck FAQ: Valve Answers the Biggest Ques...,(IGN) Steam Deck FAQ: Valve Answers the Bigges...,1626369038
4,SteamDeck,Gabe Newell: Hitting Steam Deck Price Was 'Pai...,(IGN) Gabe Newell: Hitting Steam Deck Price Wa...,1626369248
...,...,...,...,...
95,SteamDeck,My performance estimate,Its at least a RX550. As smaller GPUs are easi...,1626414616
96,SteamDeck,"Does all version have ""tempered glass""?",The descriptions states that the expensive ver...,1626414859
97,SteamDeck,Steam Link,I envision 90% of my time gaming will be at ho...,1626416365
98,SteamDeck,Early Modding Question...,Anyone know how it will handle modding games l...,1626416454


In [ ]:
steam_deck_df['cleaned_selftext'] =  steam_deck_df['selftext'].progress_apply(clean_and_spellcheck)
steam_deck_df['suicidal_text'] = steam_deck_df['cleaned_selftext'].progress_apply(pred)

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:00<00:00, 1224.51it/s]


In [ ]:
steam_deck_df[['cleaned_selftext','title','selftext','created_utc','suicidal_text']].query("suicidal_text > 0.5 and cleaned_selftext != 'removed' and cleaned_selftext != 'deleted'")

,cleaned_selftext,title,selftext,created_utc,suicidal_text
0,a place for members of r steamdeck to chat wit...,r/SteamDeck Lounge,A place for members of r/SteamDeck to chat wit...,1626368162,0.999525
7,i travel quite a bit and the switch has become...,Any information on how long it can be offline?,I travel quite a bit and the switch has become...,1626370451,1.000000
87,i remember they stopped this for the index atl...,Do you think we can purchase the Deck with ste...,I remember they stopped this for the Index (at...,1626408582,0.999997
97,i envision of my time gaming will be at home w...,Steam Link,I envision 90% of my time gaming will be at ho...,1626416365,0.996989


In [ ]:
steam_deck_df[['subreddit','title','selftext','created_utc','suicidal_text']]


1.0
